# **GENES MAPPING**

In [ ]:
###############################################
# PubMed Gene–Condition Search Automation
# Author: Sonalika’s assistant :)
###############################################

# ---- Install required packages (only runs if missing) ----
packages <- c("rentrez", "readr", "dplyr")

new_pkgs <- packages[!(packages %in% installed.packages()[, "Package"])]
if (length(new_pkgs)) {
  install.packages(new_pkgs, repos = "https://cloud.r-project.org")
}

In [ ]:
# ---- Load libraries ----
library(rentrez)
library(readr)
library(dplyr)

# ---------- CONFIG ----------
email <- "your_email@example.com"     # 👈 Replace with your email (required by NCBI)
# Optional: Set NCBI API key to increase rate limits (get one from your NCBI account)
# Sys.setenv(NCBI_API_KEY = "your_ncbi_api_key_here")
Sys.setenv(NCBI_API_KEY = Sys.getenv("NCBI_API_KEY"))

condition <- "Psoriasis"              # 👈 Change to any condition you want
genes_csv <- "/content/genes.csv"              # CSV with column "Gene"
out_csv <- "pubmed_gene_condition_hits.csv"
query_template <- '("%s"[All Fields]) AND ("%s"[All Fields])'
sleep_sec <- if (nzchar(Sys.getenv("NCBI_API_KEY"))) 0.12 else 0.35
# ---------------------------

entrez_email <- email

# ---- Read gene list ----
genes_df <- read_csv(genes_csv, show_col_types = FALSE)
if (!"Gene" %in% names(genes_df)) stop('Input CSV must have a "Gene" column.')
genes <- genes_df$Gene %>% as.character() %>% trimws() %>% .[. != ""]

# ---- Query PubMed ----
results <- lapply(seq_along(genes), function(i) {
  gene <- genes[i]
  q <- sprintf(query_template, gene, condition)
  count <- NA_integer_
  status <- "Error"
  try({
    res <- entrez_search(db = "pubmed", term = q, retmax = 0)
    count <- as.integer(res$count)
    status <- ifelse(count > 0, "Yes", "No records found")
  }, silent = TRUE)
  if (i %% 50 == 0) message(sprintf("Processed %d / %d", i, length(genes)))
  Sys.sleep(sleep_sec)
  tibble(Gene = gene, Condition = condition, Query = q,
         PubMed_Count = ifelse(is.na(count), NA, count),
         Has_Papers = status)
})

# ---- Save results ----
bind_rows(results) %>% write_csv(out_csv)
cat("✅ Done! Results saved to:", out_csv, "\n")


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘XML’



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Processed 50 / 517

Processed 100 / 517

Processed 150 / 517

Processed 200 / 517

Processed 250 / 517

Processed 300 / 517

Processed 350 / 517

Processed 400 / 517

Processed 450 / 517

Processed 500 / 517



✅ Done! Results saved to: pubmed_gene_condition_hits.csv 


In [ ]:
# ---- Load libraries ----
library(rentrez)
library(readr)
library(dplyr)

# ---------- CONFIG ----------
email <- "your_email@example.com"     # 👈 Replace with your email (required by NCBI)
# Optional: Set NCBI API key to increase rate limits (get one from your NCBI account)
# Sys.setenv(NCBI_API_KEY = "your_ncbi_api_key_here")
Sys.setenv(NCBI_API_KEY = Sys.getenv("NCBI_API_KEY"))

condition <- "Psoriasis"              # 👈 Change to any condition you want
genes_csv <- "/content/genes.csv"              # CSV with column "Gene"
out_csv <- "pubmed_UR_hits.csv"
query_template <- '("%s"[All Fields]) AND ("%s"[All Fields])'
sleep_sec <- if (nzchar(Sys.getenv("NCBI_API_KEY"))) 0.12 else 0.35
# ---------------------------

entrez_email <- email

# ---- Read gene list ----
genes_df <- read_csv(genes_csv, show_col_types = FALSE)
if (!"Gene" %in% names(genes_df)) stop('Input CSV must have a "Gene" column.')
genes <- genes_df$Gene %>% as.character() %>% trimws() %>% .[. != ""]

# ---- Query PubMed ----
results <- lapply(seq_along(genes), function(i) {
  gene <- genes[i]
  q <- sprintf(query_template, gene, condition)
  count <- NA_integer_
  status <- "Error"
  try({
    res <- entrez_search(db = "pubmed", term = q, retmax = 0)
    count <- as.integer(res$count)
    status <- ifelse(count > 0, "Yes", "No records found")
  }, silent = TRUE)
  if (i %% 50 == 0) message(sprintf("Processed %d / %d", i, length(genes)))
  Sys.sleep(sleep_sec)
  tibble(Gene = gene, Condition = condition, Query = q,
         PubMed_Count = ifelse(is.na(count), NA, count),
         Has_Papers = status)
})

# ---- Save results ----
bind_rows(results) %>% write_csv(out_csv)
cat("✅ Done! Results saved to:", out_csv, "\n")


Processed 50 / 455

Processed 100 / 455

Processed 150 / 455

Processed 200 / 455

Processed 250 / 455

Processed 300 / 455

Processed 350 / 455

Processed 400 / 455

Processed 450 / 455



✅ Done! Results saved to: pubmed_UR_hits.csv 


# **PATHWAYS MAPPING**

In [ ]:
###############################################
# PubMed Pathway ↔ Psoriasis Mapper (R / Colab)
###############################################

# ---- Install (if needed) & load packages ----
need <- c("rentrez", "readr", "dplyr", "stringr", "tibble", "purrr")
inst <- need[!(need %in% installed.packages()[, "Package"])]
if (length(inst)) install.packages(inst, repos = "https://cloud.r-project.org")

library(rentrez)
library(readr)
library(dplyr)
library(stringr)
library(tibble)
library(purrr)

# ---------- CONFIG ----------
email <- "your_email@example.com"   # NCBI requires a contact email
# Optional: boost rate limits with your API key
# Sys.setenv(NCBI_API_KEY = "your_ncbi_api_key")
sleep_sec <- if (nzchar(Sys.getenv("NCBI_API_KEY"))) 0.12 else 0.35

condition <- "Psoriasis"            # change if needed
in_csv    <- "/content/Pathways.csv"         # CSV with column "Pathway"
out_csv   <- "pubmed_pathway_psoriasis_hits.csv"

# Query templates:
Q_TA   <- '("%s"[Title/Abstract]) AND ("%s"[Title/Abstract])'
Q_ALL  <- '("%s"[All Fields]) AND ("%s"[All Fields])'
# Optionally prefer MeSH for condition:
# Q_TA <- '("%s"[Title/Abstract]) AND ("Psoriasis"[Mesh] OR psoriasis[Title/Abstract] OR psoriatic[Title/Abstract])'

entrez_email <- email
# --------------------------------------------

# ---- Load pathways ----
df <- read_csv(in_csv, show_col_types = FALSE)
if (!"Pathway" %in% names(df)) stop('Input CSV must have a "Pathway" column.')

# Clean/normalize pathway strings a bit (keep as-is for exact phrase search)
clean <- function(x) {
  x %>%
    as.character() %>%
    str_squish() %>%
    str_replace_all("[\u2018\u2019]", "'") %>%  # curly quotes → straight
    str_replace_all('[\u201C\u201D]', '"')     # curly double-quotes → straight
}

pathways <- df$Pathway %>% clean() %>% .[. != ""]

# ---- Helper to get PubMed count ----
pm_count <- function(term) {
  out <- tryCatch({
    h <- entrez_search(db = "pubmed", term = term, retmax = 0)
    as.integer(h$count)
  }, error = function(e) NA_integer_)
  out
}

# ---- Search each pathway (two-stage: TA → All) ----
results <- imap_dfr(pathways, function(pw, i) {
  # Stage 1: Title/Abstract
  q_ta  <- sprintf(Q_TA, pw, condition)
  c_ta  <- pm_count(q_ta)

  # Stage 2: fallback to All Fields if TA returned 0 or NA
  need_fallback <- is.na(c_ta) || c_ta == 0
  q_all <- if (need_fallback) sprintf(Q_ALL, pw, condition) else NA_character_
  c_all <- if (need_fallback) pm_count(q_all) else NA_integer_

  Sys.sleep(sleep_sec)
  if (i %% 50 == 0) message(sprintf("Processed %d / %d", i, length(pathways)))

  # Final decision: known if TA>0 or ALL>0
  known <- ((is.integer(c_ta) && !is.na(c_ta) && c_ta > 0) ||
            (is.integer(c_all) && !is.na(c_all) && c_all > 0))

  tibble(
    Pathway           = pw,
    Condition         = condition,
    Query_TitleAbs    = q_ta,
    Count_TitleAbs    = c_ta,
    Query_AllFields   = q_all,
    Count_AllFields   = c_all,
    Known_in_PubMed   = ifelse(known, "Yes", "No records found")
  )
})

# ---- Save ----
write_csv(results, out_csv)
cat("✅ Done. Results saved to", out_csv, "\n")

# Tip: in Colab, copy to /content for easy download (optional)
system(paste("cp", shQuote(out_csv), "/content/"))


Processed 50 / 52



✅ Done. Results saved to pubmed_pathway_psoriasis_hits.csv 


# **IPA CP, DB AND ML Pathways Mapping**

In [1]:
###############################################
# PubMed Pathway ↔ Psoriasis Mapper (R / Colab)
###############################################

# ---- Install (if needed) & load packages ----
need <- c("rentrez", "readr", "dplyr", "stringr", "tibble", "purrr")
inst <- need[!(need %in% installed.packages()[, "Package"])]
if (length(inst)) install.packages(inst, repos = "https://cloud.r-project.org")

library(rentrez)
library(readr)
library(dplyr)
library(stringr)
library(tibble)
library(purrr)

# ---------- CONFIG ----------
email <- "your_email@example.com"   # NCBI requires a contact email
# Optional: boost rate limits with your API key
# Sys.setenv(NCBI_API_KEY = "your_ncbi_api_key")
sleep_sec <- if (nzchar(Sys.getenv("NCBI_API_KEY"))) 0.12 else 0.35

condition <- "Psoriasis"            # change if needed
in_csv    <- "/content/Pathway.csv"         # CSV with column "Pathway"
out_csv   <- "pubmed_pathway_psoriasis_hits.csv"

# Query templates:
Q_TA   <- '("%s"[Title/Abstract]) AND ("%s"[Title/Abstract])'
Q_ALL  <- '("%s"[All Fields]) AND ("%s"[All Fields])'
# Optionally prefer MeSH for condition:
# Q_TA <- '("%s"[Title/Abstract]) AND ("Psoriasis"[Mesh] OR psoriasis[Title/Abstract] OR psoriatic[Title/Abstract])'

entrez_email <- email
# --------------------------------------------

# ---- Load pathways ----
df <- read_csv(in_csv, show_col_types = FALSE)
if (!"Pathway" %in% names(df)) stop('Input CSV must have a "Pathway" column.')

# Clean/normalize pathway strings a bit (keep as-is for exact phrase search)
clean <- function(x) {
  x %>%
    as.character() %>%
    str_squish() %>%
    str_replace_all("[\u2018\u2019]", "'") %>%  # curly quotes → straight
    str_replace_all('[\u201C\u201D]', '"')     # curly double-quotes → straight
}

pathways <- df$Pathway %>% clean() %>% .[. != ""]

# ---- Helper to get PubMed count ----
pm_count <- function(term) {
  out <- tryCatch({
    h <- entrez_search(db = "pubmed", term = term, retmax = 0)
    as.integer(h$count)
  }, error = function(e) NA_integer_)
  out
}

# ---- Search each pathway (two-stage: TA → All) ----
results <- imap_dfr(pathways, function(pw, i) {
  # Stage 1: Title/Abstract
  q_ta  <- sprintf(Q_TA, pw, condition)
  c_ta  <- pm_count(q_ta)

  # Stage 2: fallback to All Fields if TA returned 0 or NA
  need_fallback <- is.na(c_ta) || c_ta == 0
  q_all <- if (need_fallback) sprintf(Q_ALL, pw, condition) else NA_character_
  c_all <- if (need_fallback) pm_count(q_all) else NA_integer_

  Sys.sleep(sleep_sec)
  if (i %% 50 == 0) message(sprintf("Processed %d / %d", i, length(pathways)))

  # Final decision: known if TA>0 or ALL>0
  known <- ((is.integer(c_ta) && !is.na(c_ta) && c_ta > 0) ||
            (is.integer(c_all) && !is.na(c_all) && c_all > 0))

  tibble(
    Pathway           = pw,
    Condition         = condition,
    Query_TitleAbs    = q_ta,
    Count_TitleAbs    = c_ta,
    Query_AllFields   = q_all,
    Count_AllFields   = c_all,
    Known_in_PubMed   = ifelse(known, "Yes", "No records found")
  )
})

# ---- Save ----
write_csv(results, out_csv)
cat("✅ Done. Results saved to", out_csv, "\n")

# Tip: in Colab, copy to /content for easy download (optional)
system(paste("cp", shQuote(out_csv), "/content/"))


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘XML’



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Processed 50 / 73



✅ Done. Results saved to pubmed_pathway_psoriasis_hits.csv 
